# 2 – RAG with Propositions

Same pipeline as Example 1, but we change **what** we index.

Instead of embedding raw chunks, an LLM first rewrites every chunk into *propositions*: short, self-contained statements that each carry exactly one fact (Chen et al., 2023, [*Dense X Retrieval*](https://arxiv.org/abs/2312.06648)).

> **Chunk:** "Its balance sheet remained solid. Net debt fell to EUR 2.1 billion, and the leverage ratio improved from 2.8x to 2.3x."
>
> **Propositions:**
> - "Novaterra Energy's net debt fell to EUR 2.1 billion in 2025."
> - "Novaterra Energy's leverage ratio improved from 2.8x to 2.3x in 2025."

Two things happened:

- one chunk became several **fine-grained** units – a precise question matches one precise vector instead of a paragraph-sized average;
- pronouns were **resolved** ("Its" → "Novaterra Energy's") – every unit makes sense on its own, so the embedding knows which company it is about.

The cost: one LLM call per chunk at indexing time. We cache the result to disk so we pay only once.

We keep a link from every proposition back to its source chunk (its "parent"). At query time we can give the LLM either the propositions or the parents.

## Imports

Two additions compared with Example 1: `json` for the proposition cache, and `BaseModel` / `Field` from Pydantic to describe the structured output we want from the LLM.

In [ ]:
import json
import os
from pathlib import Path

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

## Configuration

- `LLM_MODEL` / `EMBEDDING_MODEL` – the same Ollama-served models as in Example 1.
- `CACHE_FILE` – where the extracted propositions are stored. Delete it to run the LLM pass again, for example after you edit the prompt or switch to another LLM.
- `CHUNK_OVERLAP = 0` – no overlap is needed: propositions are self-contained.
- `TOP_K = 6` – propositions are small, so we can afford to retrieve more of them.
- `USE_PARENT_CONTEXT`
  - `True` → send the *parent chunks* of the matched propositions to the LLM (precise matching, but richer context for the answer)
  - `False` → send only the propositions themselves (minimal tokens)

In [ ]:
DATA_DIR = Path("data")
CACHE_FILE = Path("propositions_cache.json")

LLM_MODEL = os.getenv("RAG_LLM", "ollama:mistral-small3.2:24b")
EMBEDDING_MODEL = "embeddinggemma:300m"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 0
TOP_K = 6

USE_PARENT_CONTEXT = True

print("LLM:", LLM_MODEL)

## Steps 1 + 2 – LOAD and CHUNK (as in Example 1)

Two small differences:

- The first line of each file is its title. We keep it in `metadata["title"]`, so the proposition LLM knows which entity "the company" refers to even when the chunk itself never mentions the name.
- Every chunk gets a stable `chunk_id`, so propositions can point back to it.

In [ ]:
docs = []
for path in sorted(DATA_DIR.glob("*.txt")):
    text = path.read_text(encoding="utf-8")
    docs.append(Document(
        page_content=text,
        metadata={"source": path.name, "title": text.splitlines()[0]},
    ))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
    is_separator_regex=True,
)
chunks = splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
print(f"Loaded {len(docs)} documents -> {len(chunks)} chunks")

## Step 2b – PROPOSITIONS (the new part)

### The output schema

We ask the LLM for *structured output*: it must return JSON that matches this Pydantic model, and LangChain parses it into a Python object for us – no string parsing. The field description is sent to the model as part of the schema, so it doubles as an instruction.

In [ ]:
class Propositions(BaseModel):
    propositions: list[str] = Field(
        description="Atomic, self-contained factual statements."
    )

### The prompt

The rules follow the *Dense X Retrieval* paper. Rule 3 (decontextualize) is what makes propositions work: a statement that still says "it" or "the company" would be just as ambiguous as the original chunk. We pass the document title so the LLM can resolve those references even when the passage never names the entity.

In [ ]:
PROPOSITION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Decompose the passage into clear and simple propositions.\n"
     "Rules:\n"
     "1. Split compound sentences into simple sentences, one fact each.\n"
     "2. Keep the original wording where possible.\n"
     "3. Decontextualize: replace pronouns and vague references "
     "('it', 'the company', 'this policy') with the full entity name, "
     "and add any context needed for the statement to be understood alone.\n"
     "4. Include numbers, units, dates and conditions exactly as stated.\n"
     "5. Skip headings and text with no factual content."),
    ("human", "Document title: {title}\n\nPassage:\n{passage}"),
])

### Running the extraction (one LLM call per chunk)

This is the expensive part of indexing, so the result is cached in `CACHE_FILE` and reused on later runs.

- `llm.with_structured_output(Propositions)` wraps the chat model so that it returns a `Propositions` instance instead of a free-text message.
- `PROPOSITION_PROMPT | ...` is a small LCEL chain: fill the prompt, call the model, parse the JSON.
- `.batch()` sends the calls concurrently (`max_concurrency=5`) – much faster than a `for` loop against a hosted API. A local Ollama server queues requests for the same model and works through them one at a time unless `OLLAMA_NUM_PARALLEL` is raised, so expect this cell to take a minute or two the first time.

The cache is a plain list of `{"chunk_id": ..., "propositions": [...]}` records.

In [ ]:
if CACHE_FILE.exists():
    print(f"Using cached propositions from {CACHE_FILE.name}")
    raw = json.loads(CACHE_FILE.read_text())
else:
    llm = init_chat_model(LLM_MODEL, temperature=0)
    extractor = PROPOSITION_PROMPT | llm.with_structured_output(Propositions)

    inputs = [{"title": c.metadata["title"], "passage": c.page_content}
              for c in chunks]
    results = extractor.batch(inputs, config={"max_concurrency": 5})

    raw = [
        {"chunk_id": c.metadata["chunk_id"], "propositions": r.propositions}
        for c, r in zip(chunks, results)
    ]
    CACHE_FILE.write_text(json.dumps(raw, indent=2))

print(f"{sum(len(item['propositions']) for item in raw)} propositions "
      f"from {len(raw)} chunks")

### From strings to Documents

Every proposition becomes its own `Document`. `metadata["chunk_id"]` is the pointer back to the parent chunk; `metadata["source"]` is copied from the parent so we can still show where a fact came from.

The printout shows one chunk and what it became. Note how the pronouns were replaced by the full entity name.

In [ ]:
by_id = {c.metadata["chunk_id"]: c for c in chunks}
prop_docs = []
for item in raw:
    parent = by_id[item["chunk_id"]]
    for text in item["propositions"]:
        prop_docs.append(Document(
            page_content=text,
            metadata={"chunk_id": item["chunk_id"],
                      "source": parent.metadata["source"]},
        ))

print(f"Extracted {len(prop_docs)} propositions from {len(chunks)} chunks")
example_id = min(4, len(chunks) - 1)
print("Example - chunk:\n   ",
      chunks[example_id].page_content[:200].replace("\n", " "), "...")
print("  became:")
for p in [d for d in prop_docs if d.metadata["chunk_id"] == example_id][:4]:
    print("    -", p.page_content)

## Steps 3 + 4 – EMBED and STORE the propositions (not the chunks!)

Same embedding model and vector store as Example 1; only the input differs. `reset_collection()` keeps the cell safe to re-run (see Example 1).

In [ ]:
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
vector_store = Chroma(
    collection_name="propositions",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)
vector_store.reset_collection()
ids = vector_store.add_documents(prop_docs)
print(f"Indexed {len(ids)} propositions")

## Step 5 – RETRIEVE + GENERATE

The similarity search now returns propositions. What we send to the LLM depends on `USE_PARENT_CONTEXT`:

- `True` – look up the parent chunk of every hit. Several propositions often come from the same chunk, so the ids are de-duplicated with `dict.fromkeys(...)`, which keeps the order of the best match.
- `False` – send the matched propositions themselves.

The answer prompt is the same grounded prompt as in Example 1.

In [ ]:
PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions using ONLY the context below. "
     "If the context does not contain the answer, say you don't know. "
     "Cite the context items you used like [1], [2].\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])


def answer(question, vector_store, chunks, llm):
    hits = vector_store.similarity_search_with_relevance_scores(question, k=TOP_K)

    print(f"\nTop {TOP_K} propositions for: {question!r}")
    for rank, (doc, score) in enumerate(hits, start=1):
        print(f"  {rank}. score={score:.3f}  (chunk {doc.metadata['chunk_id']})  "
              f"{doc.page_content}")

    if USE_PARENT_CONTEXT:
        parent_ids = list(dict.fromkeys(d.metadata["chunk_id"] for d, _ in hits))
        context_texts = [chunks[i].page_content for i in parent_ids]
        print(f"  -> sending {len(context_texts)} parent chunks to the LLM")
    else:
        context_texts = [d.page_content for d, _ in hits]

    context = "\n\n".join(f"[{i}] {t}" for i, t in enumerate(context_texts, 1))
    chain = PROMPT | llm | StrOutputParser()
    return chain.invoke({"context": context, "question": question})

## Run it

The questions are chosen so that the answer is a single precise fact buried in a longer chunk.

In [ ]:
QUESTIONS = [
    "What was Novaterra's leverage ratio at the end of 2025?",
    "What DSTI limit applies to an applicant earning EUR 1,800 per month?",
    "How much did regenerative braking reduce tram electricity use?",
]

llm = init_chat_model(LLM_MODEL, temperature=0)
for q in QUESTIONS:
    print("\nAnswer:", answer(q, vector_store, chunks, llm))
    print("-" * 80)

## Things to try

1. Change `USE_PARENT_CONTEXT` in the configuration cell, re-run it and the cell above, and compare the answers and the amount of context sent to the LLM.
2. Edit the proposition prompt, delete `propositions_cache.json`, and re-run from *Step 2b*. How do the propositions change?
3. Ask the same question in Example 1 and here. Which retrieval hits are more precise?